In [1]:
from stormpy import export_to_drn
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

from verimon.logger import setup_logging

setup_logging()

In [2]:
from verimon import loaders
from random import randrange
from math import sqrt

gb_e = "../tests/gb_exact.pm"
gb = loaders.load_dfa(gb_e)

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

def random_ladder(n):
    source = randrange(1,n-int(sqrt(n)))
    dest = randrange(source+int(sqrt(n)), int(min(n, source + n / 2)))
    return source, dest

def random_snake(n):
    source = randrange(int(sqrt(n))+2,n)
    dest = randrange(1, source-int(sqrt(n)))
    return source, dest

n = 5**2
ladders = {0:0}
snakes = {0:0}
while not set(ladders.keys()).isdisjoint(set(snakes.keys())):
    ladders = dict(random_ladder(n) for _ in range(int(sqrt(n))))
    snakes = dict(random_snake(n) for _ in range(int(sqrt(n))))

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [3]:
from stormvogel.mapping import stormpy_to_stormvogel, stormvogel_to_stormpy
from stormvogel.show import show

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
show(mc_sv)

Output()

Output()

In [4]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

player_path = [(0, [])]

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in mc_sv.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

<IPython.core.display.Javascript object>

In [5]:
from aalpy import run_Lstar
from verimon.MonitorLearning import FilteringSUL, VerimonEqOracle

setup_logging()


threshold = 0.4
slack = 0.1
horizon = 10
relative_error = 0.1
spec = 'P>0.5 [F<3 "good" ]'

alphabet = ["init", "normal", "snake", "ladder"]

filtering_sul = FilteringSUL(
    mc, 
    "init", 
    alphabet, 
    spec, 
    threshold, 
    horizon
)
eq_oracle = VerimonEqOracle(
    alphabet,
    filtering_sul,
    mc_sv,
    gb,
    threshold,
    slack,
    horizon,
    spec,
    'good',
    relative_error
    
)
learned_monitor = run_Lstar(
    alphabet,
    filtering_sul,
    eq_oracle,
    automaton_type="dfa",
    print_level=2,
)

Hypothesis 1: 1 states.
Visualization started in the background thread.
DEBUG:2024-10-28 13:46:19,668 - (0.00s) - MonitorLearning.py - Finding false negative probability 
DEBUG:2024-10-28 13:46:19,669 - (0.00s) - verify.py - Building model 
DEBUG:2024-10-28 13:46:19,670 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-10-28 13:46:19,673 - (0.00s) - verify.py - Pruning done 
INFO:2024-10-28 13:46:19,678 - (0.00s) - MDP_product.py - New good states become: [['[pos=25]']] 
DEBUG:2024-10-28 13:46:19,679 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-28 13:46:19,680 - (0.00s) - MDP_product.py - Finished product setup, created 10 observations 
DEBUG:2024-10-28 13:46:19,689 - (0.01s) - MDP_product.py - Created product 522 states 
DEBUG:2024-10-28 13:46:19,702 - (0.01s) - MDP_product.py - Created product transitions for 520 states 
DEBUG:2024-10-28 13:46:19,704 - (0.00s) - verify.py - creating product done 
DEBUG:2024-10-28 13:46:19,704 - (0.00s) - verify.py - Creating trace 
Model s

In [9]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.unrolling import simulator_unroll, prune_monitor
from verimon.algs import complement_model

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
# show(mon_cycl)
complement_model(mon_cycl, "accepting")
mon = simulator_unroll(mon_cycl, horizon)
prune_monitor(mon)
print(len(mon.states))
show(mon)

129


Output()

Output()

In [7]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.verify import *

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
export_to_drn(stormvogel_to_stormpy(mon_cycl), "monitor.drn")
result_goal, trace, assignment, product = false_positive(mc_sv, mon_cycl, gb, horizon, options = {"good_spec": spec})

Write to file monitor.drn.
DEBUG:2024-10-28 13:46:29,704 - (0.31s) - verify.py - Building model 
DEBUG:2024-10-28 13:46:29,707 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-10-28 13:46:29,727 - (0.02s) - verify.py - Pruning done 
INFO:2024-10-28 13:46:29,736 - (0.01s) - MDP_product.py - New good states become: [['[pos=25]']] 
DEBUG:2024-10-28 13:46:29,736 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-28 13:46:29,737 - (0.00s) - MDP_product.py - Finished product setup, created 12 observations 
DEBUG:2024-10-28 13:46:29,824 - (0.09s) - MDP_product.py - Created product 2550 states 
DEBUG:2024-10-28 13:46:29,947 - (0.12s) - MDP_product.py - Created product transitions for 2548 states 
DEBUG:2024-10-28 13:46:29,948 - (0.00s) - verify.py - creating product done 
DEBUG:2024-10-28 13:46:29,948 - (0.00s) - verify.py - Creating trace 
> progress 0.637%, elapsed 3 s, estimated 471 s, iters = {MDP: 167}, opt = 0.546
> progress 3.5%, elapsed 6 s, estimated 172 s, iters = {MDP: 533}, o

In [8]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

# player_path = [(0, [])]
poss = product.simulate_paynt_assignment(assignment, 100000)
player_path = poss

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in product.mc.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

s0, labels=!s0 !l0 !g0 init [pos=0]

	--[0, 3:ladder]-->
s159, labels=!g0 !l3 !s1 [pos=1] ladder
	--[1, 3:ladder]-->
s425, labels=!g0 !l8 !s7 [pos=9] ladder
	--[2, 1:normal]-->
s743, labels=!g0 !l14 !s13 [pos=14] normal
	--[3, 3:ladder]-->
s1061, labels=!g0 !l20 !s19 [pos=19] ladder
	--[4, 1:normal]-->
s1430, labels=!g0 !l27 !s24 [pos=24] normal
	--[5, 1:normal]-->
s1793, labels=!g0 !l34 !s23 [pos=23] normal
	--[6, 1:normal]-->
s2106, labels=!g0 !l40 !s24 [pos=24] normal
	--[8, 4:end]-->
s2, labels=stop
it took 30 tries until the goal was reached


<IPython.core.display.Javascript object>